In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<b><font size="5" color="red" >ch11_데이터프레임과 시리즈(Pandas)</font></b>
- pip install pandas / conda install pandas (아나콘다 프롬프트)
# 1절. 판다스 패키지
- 데이터 분석을 위해 반드시 알아야 할 패키지. 넘파이 기반으로 다른 많은 라이브러리와 잘 통합되도록 설계
- 1차원 구조를 갖는 시리즈, 2차원 구조를 갖는 데이터프레임(excel의 스프레드 시트)을 제공
- 판다스 장점 : 파일io, 부분 데이터 추출, 크기변경, 데이터 분할, 병합, 정렬, 결측치 처리, 데이터 분할, 피벗과 언피벗(와이드포맷과 롱포맷)에 용이

- [Pandas API reference](https://pandas.pydata.org/docs/reference/index.html)


In [ ]:
import pandas as pd
pd.__version__

In [ ]:
data = pd.read_csv('data/ch09_member1.csv', encoding='utf-8')
display(data)
type(data)

In [ ]:
data = pd.read_csv('data/ch09_member1.csv',
                   header=None,
                  names=['name','age','email','address'])
data.head(2)

In [ ]:
data = pd.read_json('data/ch09_member.json', encoding='utf-8') # 기본값이 utf-8
data

# 2절. 데이터 프레임 만들기
## 2.1 딕셔너리를 이용해서 데이터프레임 만들기

In [ ]:
d = {'kor':[100,90], 'mat':[95,99]} # 딕셔너리를 프레임으로
df = pd.DataFrame(data=d)
df

In [ ]:
df.to_dict() # 데이터 프레임을 딕셔너리로 변환

In [ ]:
df.to_numpy()

In [ ]:
# 딕셔너리 리스트를 데이터 프레임으로
d = [{'kor':100, 'mat':95}, {'kor':90, 'mat':99}]
df = pd.DataFrame(data=d)
df

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
d = [{'kor':100, 'mat':100}, {'kor':90, 'mat':91}, {'kor':93, 'math':90}]
df = pd.DataFrame(data=d)
# 결측치는 실수형(float64). 정수형변환 불가(결측치대체를 한후 정수형 변환)
df

In [ ]:
df.info()

## 리스트를 이용해서 데이터 프레임 만들기

In [ ]:
title = ['책1', '책2', '책3']
price = [15000, 18000, 10000]
df = pd.DataFrame(data={'title':title, 'price':price})
df

In [ ]:
import numpy as np
np.c_[title,price]

In [ ]:
df = pd.DataFrame(np.c_[title,price], columns=['책이름','가격'])
df

In [ ]:
l = [['책1', 15000],
     ['책2', 18000],
     ['책3', 10000]]
df = pd.DataFrame(l, columns=['책이름','가격'])
df

## 2.3 read_csv

In [ ]:
%ls C:/ai_x/download/shareData/상가정보_250331/

In [ ]:
df = pd.read_csv(r'C:/ai_x/download/shareData/상가정보_250331/소상공인시장진흥공단_상가(상권)정보_세종_202503.csv',
                 encoding='utf-8',
                 sep=',',
                 low_memory=False, # 데이터 용량이 클 경우만 기입
                 nrows=30 # 처음 30행만
                 )
df.shape

In [ ]:
df.head(1)

In [ ]:
# 판다스 디스플레이 옵션
pd.options.display.max_columns = 39 # 기본값 : 20
pd.options.display.max_rows = 40
df.head(1)

In [ ]:
df.head(1).T

In [ ]:
# (1) CSV 파일 불러오기(기본값)
# encoding = utf-8, sep = ',', csv파일의 첫번째 줄을 header, #이 있는 줄도 데이터로 인식
member = pd.read_csv('data/ch11_member.csv')
member

In [ ]:
member.info()

In [ ]:
# 형변환 Age열:int64 -> int16
member['Age'] = member['Age'].astype(np.int16) # 'int' : np.int32
member.info()

In [ ]:
# 형변환 Birth열 : object(문자) -> datetime64
member['Birth_as']=member['Birth'].astype('datetime64')
member.info()

In [ ]:
member['Birth_to'] = pd.to_datetime(member['Birth']) # astype 함수보다 안전한 형변환
member.info()

In [ ]:
# (2) 특정행 제외하고 csv파일읽기
member = pd.read_csv('data/ch11_membercp949.csv',
                     encoding='cp949',
                     skiprows=[1, 6]) # 1,6번째 행은 제외하고 읽어오기
member

In [ ]:
# (3) 주석(#)제외, datetime형 필드 지정하여 읽기
member = pd.read_csv('data/ch11_membertab.csv',
                     sep='\t',
                     comment='#',
                     parse_dates=['Birth']) # datetime형으로 읽어올 필드 지정
member.info()

In [ ]:
# (4) sep='|', 상위 5행만 읽어오기
member = pd.read_csv('data/ch11_membersep.csv',
                     sep='|',
                     nrows=5)
member

## 2.4 패지키에서 가져오기 : iris 데이터
- iris 가져오기 방법1 : sklearn (머신러닝 패키지)
- iris 가져오기 방법2 : statsmodels (R 데이터)
- iris 가져오기 방법3 : seaborn(시각화 패키지)
### iris 가져오기 방법1 : sklearn (머신러닝 패키지)


In [ ]:
from sklearn import datasets
# sklearn : 머신러닝 패키지(전처리함수, 머신러닝 함수, 성능평가를 위한 함수, 학습데이터셋)
iris = datasets.load_iris()
iris.keys() # 딕셔너리 형태


In [ ]:
print(iris.DESCR) # iris 데이터에 대한 설명

In [ ]:
iris.data # 독립변수
iris['data'][:3]

In [ ]:
iris.feature_names # 독립변수의 열이름
iris['feature_names']

In [ ]:
# 데이터 프레임에서 열이름
columns = [ col[:-5].replace(' ', '_') for col in iris.feature_names]+['species']
columns

In [ ]:
iris.target
iris['target'] # 머신러닝시 종속변수

In [ ]:
iris.target_names

In [ ]:
iris.target_names[iris.target]

In [ ]:
# 독립변수
data = iris.data
# 종속변수
target = iris.target_names[iris.target]
data.shape, target.shape # 독립변수와 종속변수의 차원

In [ ]:
# 독립변수와 종속변수를 stack한 후 데이터 프레임으로 완성
pd.DataFrame(data = np.hstack( (data, target.reshape(-1,1))),
             columns=columns)

### iris 가져오기 방법2 : statsmodels (R데이터)

In [ ]:
# R언어에 있는 유명한 데이터셋을 가져오는 함수
from statsmodels.datasets import get_rdataset
iris_dataset = get_rdataset('iris',
                            package='datasets',
                            cache=True) # 한번 다운로드한 데이터 셋을 내 PC에 저장
iris_dataset

In [ ]:
iris = iris_dataset.data
iris.head(1)

In [ ]:
iris.columns = [col.lower().replace('.','_') for col in iris.columns]
iris.head(1)

### iris 가져오기 방법3 : seaborn(시각화 패키지)

In [ ]:
import seaborn as sns
iris = sns.load_dataset('iris')
iris.head(1)

In [ ]:
# 데이터 프레임을 csv파일로 저장
iris.to_csv('data/ch11_iris.csv',
            # sep=',' , encoding='utf-8', #기본값
            index=False) # 행이름을 제외하고 출력

In [ ]:
import pandas as pd
load_iris = pd.read_csv('data/ch11_iris.csv')
load_iris.head(1)

In [ ]:
# 데이터 프레임을 압축파일로(.zip, .gz, .bz2)
iris.to_csv('data/ch11_iris.gz',
            index = False,
            compression='infer')

In [ ]:
load_iris = pd.read_csv('data/ch11_iris.gz',
                         compression='infer')
load_iris

# 3절. 이름(열, 행) 지정하기
## 3.1 열이름 지정하기

In [ ]:
member = pd.read_csv('data/ch11_member.csv',
                     comment='#',
                     parse_dates=['Birth'])
member

In [ ]:
member.columns = ['이름', '나이', '메일', '주소', '생년월일']
member.columns.name = None
member

## 3.2 행이름 지정

In [ ]:
member.index

In [ ]:
member.index = range(11,16)
member

In [ ]:
member.index = ['동','서','남','북','중']
member

In [ ]:
print(member.index.name)

In [ ]:
member.loc['남', '이름'] # loc을 이용하여 행이름과 열이름으로 데이터 부분 조회

In [ ]:
# 특정열을 index로 setting
# member를 수정하려면 (1)할당 (2)inplace 매개변수에 True (기본값은 False)
member1 = member.set_index('이름') # 이름열이 index로 setting
member1.loc['홍길동']

In [ ]:
member.set_index('이름', inplace=True) # 이름열을 index로

In [ ]:
member.head(1)

In [ ]:
member = member.reset_index(drop=True) # index를 컬럼(열)로
member

In [ ]:
member

In [ ]:
member.loc[1, '생년월일']

In [ ]:
# 시리즈(날짜임).dt :datetime열의 날짜 및 시간에 관련된 정보만 추출
member['생년월일'].dt.year

In [ ]:
member['생년월일'].dt.weekday # 0:월, 1:화, ... 5:토

## 3.3 레벨 이름 지정하기

In [ ]:
member.columns = [['기본정보', '기본정보', '기본정보', '추가정보', '추가정보'],
                  ['이름','나이', '메일', '주소', '생년월일']]
member.columns.names = ['대분류','소분류']
member

In [ ]:
member.index = [['좌우','좌우','상하','상하','상하'],
                ['동','서','남','북','중']]
member.index.names = ['레벨1','레벨2']
member

In [ ]:
member.loc[('좌우','동'),'기본정보']

# 4절. 부분 데이터 조회

In [ ]:
member

In [ ]:
member = pd.read_csv('data/ch11_member.csv', comment='#',parse_dates=['Birth'])
member

## 4.1 열 조회

In [ ]:
member['Name']
member.Name

In [ ]:
member[ ['Name','Email'] ]

In [ ]:
# member[0] 행을 조회할 경우 loc, iloc. []안에는 열이름과 조건만 사용가능

## 4.2 loc을 이용한 조회
- df.loc[행이름, 열이름] : 행이름과 열이름으로 조회
    * 행이름과 열이름 자리에 list ex.['Name','Email']
    * 행이름과 열이름 자리에 슬라이싱 from:to : from부터 to까지(from, to 포함)
    * ,열이름 생략시 모든 열

In [ ]:
member.index = ['동','서','남','북','중']
member

In [ ]:
# member 동행부터 남행
member.loc['동':'남']

In [ ]:
# member 동행부터 남행, 'Name' , 'Age' , 'Email'
member.loc['동':'남', 'Name':'Email']

In [ ]:
# member 동행과남행, 'Name' , 'Email', 'Address'
member.loc[['동','남'],['Name','Email','Address']]

In [ ]:
# loc을 이용한 특정 열 조회
member.loc[:,['Name','Address']] # member['Name','Address'] 동일

In [ ]:
member.loc['동'] # 결과가 1차원 => 시리즈

In [ ]:
member.loc['동':'동'] # 슬라이싱의 경우 결과가 2차원 => 데이터프레임

## 4.2 iloc을 이용한 조회
- df.iloc[행번호, 열번호] : 행번호와 열번호로 조회
    * 행번호과 열번호 자리에 list ex.[0,2]
    * 행번호과 열번호 자리에 슬라이싱 from:to:by : from부터 by씩 증감하면서 to앞까지 (to 미포함)
    * ,열번호 생략시 모든 열

In [ ]:
# 0번째~2번째 행 모든 열
member.iloc[0:3] # 열번호 생략시 모든 열

In [ ]:
member.iloc[0] # 결과가 1차원이면 시리즈, 데이터프레임으로 하고자 하면 슬라이싱 이용하거나
               # pd.DataFrame() 함수 이용

In [ ]:
# 짝수번째 행의 맨 마지막 열을 제외한 데이터 추출
member.iloc[::2, :-1]

In [ ]:
# 모든 행의 마지막 열만 데이터프레임으로 추출
member.iloc[:,-1:] # member.iloc[:,-1] 1차원

In [ ]:
# 0번째, 3번째행   0번째, 3번째, 4번째 열
member.iloc[0:4:3, [0,3,4]]

## 4.4 조건으로 조회
- df[조건] : 조건에 맞는 행(모든 열)
- df.loc[조건, 열이름] 또는 df[조건][열이름] : 조건에 맞는 행의 특정 열

In [ ]:
member

In [ ]:
member.Age > 22

In [ ]:
# Age가 22보다 큰 데이터 셋
member[member.Age > 22][['Name','Age']]

In [ ]:
# Age가 22보다 큰 데이터의 name과 Age
member.loc[member.Age > 22, ['Name','Age'] ]

In [ ]:
address1 = '서울시 강동구'
address2 = '부산시 중구'
print(address1.startswith('서울시'))
print(address2.startswith('서울시'))
print(address1.find('강동구')!=-1)
print(address2.find('강동구')!=-1)
print(address2.count('강동구')>0)

- 교안 pdf 29p. 시리즈에 문자함수를 쓰기 위해 참조:
https://pandas.pydata.org/pandas-docs/stable/reference/series.html#string-handling

In [ ]:
member.Address.str.startswith('서울시')

In [ ]:
# Address 가 '서울시'로 시작하는 행
member[member.Address.str.startswith('서울시')]

In [ ]:
# Address에 '강동구'가 포함된 행
display(member[member.Address.str.count('강동구')>0])
member[member.Address.str.find('강동구')!=-1]
member[member.Address.str.contains('강동구')]

In [ ]:
# Address에 '강동구'가 포함된 'Name', 'Age' 열 조회
member[member.Address.str.contains('강동구')][['Name',"Age"]]
member.loc[member.Address.str.contains('강동구'), 'Name':'Age']

In [ ]:
# 1999년도 태어난 데이터
member[member.Birth.dt.year==1999]

※ 데이터 프레임의 부분 데이터 조회 방법
- df[열이름] - 특정 열 모든 행조회
- df[조건] - 조건에 맞는 모든 행 조회
- df.loc[행이름, 열이름] : 이름조회, 조건(행이름 자리)으로 조회 가능
    * 열이름을 생략하면 모든 열
    * 행이름, 열이름 자리에 list, 슬라이싱
- df.iloc[행번호, 열번호] : 번호 조회
    * 열번호를 생략하면 모든 열
    * 행번호, 열번호 자리에 list, 슬라이싱

In [ ]:
# 1. sepal_length 열만 출력

# 2. 0~10행까지 마지막 열을 제외한 데이터 (loc, iloc)

# 3. 3~10 행중에 'sepal_length'와 'petal_length' 열만 (loc, iloc)

# 4. 0번째, 50번째, 100번째 행의 모든 열 (loc, iloc)

# 5. 0번째, 25, 50번째, 75번, 100번째, 125번째 행의 petal_length와 petal_width (loc, iloc)

# 6. species가 versicolor인 데이터의 모든 열

# 7. species가 setosa인 데이터 최초 5개 행만 출력

# 8. sepal_length가 6.5이상인 데이터 최초 5개 행만 출력

# 9. sepal_length가 7.2이상인 데이터의 'sepal_length'와 'sepal_width'와 'petal_length'

# 10. versicolor종중에서 sepal_length가 6.5보다 큰 데이터의 모든 열

In [ ]:
import seaborn as sns
iris = sns.load_dataset('iris')
iris.head(1)

In [ ]:
# 1. sepal_length 열만 출력
iris['sepal_length']

In [ ]:
# 2. 0~10행까지 마지막 열을 제외한 데이터 (loc, iloc)
iris.iloc[0:10,:-1]

In [ ]:
# 3. 3~10 행중에 'sepal_length'와 'petal_length' 열만 (loc, iloc)
iris.iloc[3:11,[0,2]]

In [ ]:
# 4. 0번째, 50번째, 100번째 행의 모든 열 (loc, iloc)
iris.iloc[0::50]
iris.loc[[0,50,100]]

In [ ]:
# 5. 0번째, 25, 50번째, 75번, 100번째, 125번째 행의 petal_length와 petal_width (loc, iloc)
iris.iloc[0::25,[2,3]]
iris.iloc[::25,2:-1]

In [ ]:
# 6. species가 versicolor인 데이터의 모든 열
iris[iris.species.str.startswith('versicolor')]

In [ ]:
# 7. species가 setosa인 데이터 최초 5개 행만 출력
iris[iris.species.str.startswith('versicolor')].head()

In [ ]:
# 8. sepal_length가 6.5이상인 데이터 최초 5개 행만 출력
iris.loc[iris.sepal_length>=6.5].head()

In [ ]:
# 9. sepal_length가 7.2이상인 데이터의 'sepal_length'와 'sepal_width'와 'petal_length'
iris.loc[iris.sepal_length>=7.2][['sepal_length','sepal_width','petal_length']]

In [ ]:
# 10. versicolor종중에서 sepal_length가 6.5보다 큰 데이터의 모든 열
iris_ver = iris[iris.species.str.startswith('versicolor')]
iris_ver.loc[iris.sepal_length>=6.5]

In [ ]:
# 1. sepal_length 열만 출력
iris['sepal_length'], iris.sepal_length

# 2. 0~10행까지 마지막 열을 제외한 데이터 (loc, iloc)
iris.loc[:10, 'sepal_length':'petal_width']
iris.iloc[:11, :-1]

# 3. 3~10 행중에 'sepal_length'와 'petal_length' 열만 (loc, iloc)
iris.loc[3:10, ['sepal_length','petal_length'] ]  
iris.iloc[3:11, [0,2] ]  

# 4. 0번째, 50번째, 100번째 행의 모든 열 (loc, iloc)
iris.loc[[0,50,100]]
iris.loc[:100:50] # 비추
iris.iloc[:101:50]

# 5. 0번째, 25, 50번째, 75번, 100번째, 125번째 행의 petal_length와 petal_width (loc, iloc)
iris.loc[[0,25,50,75,100,125], 'petal_length':'petal_width']
iris.loc[::25, 'petal_length':'petal_width'] # 비추
iris.iloc[::25, 2:-1]

# 6. species가 versicolor인 데이터의 모든 열
iris[iris.species!='versicolor']

# 7. species가 setosa인 데이터 최초 5개 행만 출력
iris[iris.species=='setosa'].head()
iris[iris.species=='setosa'].iloc[:5]

# 8. sepal_length가 6.5이상인 데이터 최초 5개 행만 출력
iris[iris.sepal_length>=6.5].head()
iris[iris.sepal_length>=6.5].iloc[:5]

# 9. sepal_length가 7.2이상인 데이터의 'sepal_length'와 'sepal_width'와 'petal_length'
iris.loc[iris.sepal_length >= 7.2, 'sepal_length':'petal_length']
iris[iris.sepal_length >= 7.2][['sepal_length','sepal_width','petal_length']]

# 10. versicolor종중에서 sepal_length가 6.5보다 큰 데이터의 모든 열
iris[ (iris.species=='versicolor') & (iris.sepal_length>6.5)]

In [ ]:
import numpy as np
np.logical_and(True, True)

# 5절. 데이터 추가 및 삭제
## 5.1 데이터 프레임의 요소 삭제
- df.drop(행이름이나 열이름, axis) : axis=0 : 행삭제 / axis=1 : 열삭제

In [ ]:
member = pd.read_csv('data/ch11_member.csv', comment='#')
member.index = ['동','서','남','북','중']
member

In [ ]:
# (1) 단일 행 삭제
member.drop('중') # axis=0 이 기본값

In [ ]:
# '동' 행이 없으면 에러
member.drop('동', inplace=True)

In [ ]:
# (2) 복수행 삭제
member.drop(['서','남'])

In [ ]:
# (3) 열 삭제
member.drop(['Age','Birth'], axis=1)

## 5.2 데이터프레임의 요소 추가

In [ ]:
# 데이터프레임에 열 추가 1 : 공통된 값으로 추가
member['favorite_no'] = 7
member

In [ ]:
member.info()

In [ ]:
# 데이터 프레임에 열 추가하는 방법 2 : 리스트로 추가
member['fn'] = [7,7,7,None] # 결측치 : None, np.nan
member

In [ ]:
import math
np.nan, None, math.nan

In [ ]:
# 결측치는 정수형 변환 불가 ( 결측치 대체 후 형변환)
# member['fn'].astype('int)

In [ ]:
# 데이터 프레임에 열추가 3 : 시리즈로 추가
member = pd.read_csv('data/ch11_member.csv', comment='#')
member['fn2'] = pd.Series([7,5,7])
member

In [ ]:
member['fn'] = pd.Series(['test','test2','test3'], index=[0,3,4])
member

In [ ]:
member.info()

In [ ]:
member

In [ ]:
# 행 추가시 추가할 데이터를 데이터프레임 -> 기존DF, 추가할 DF 연결
member = pd.read_csv('data/ch11_member.csv')
new_member = pd.DataFrame([{
    'Name':'홀길숙',
    'age' : 30,
    'Email':'h@h.com',
    'Address':'설',
    'Birth':'2010-01-01'

}])
new_member

In [ ]:
# 행 추가시 member와 new_member 연결
member = pd.concat([member, new_member])
member

In [ ]:
pd.concat([member, new_member], axis=1) # axis = 0 : 기본값 (행연결), axis = 1 : 열연결

In [ ]:
# 인덱스 재조정
member.reset_index(drop=True) # 기존의 index를 drop하고 새로운 연속된 index로 재조정
# member.reset_index() : 기존의 index를 컬럼에 편입시키고 새로운 연속된 index를 생성
member

# 6절. 병합과 연결
## 6.1 merge()를 이용한 병합

In [ ]:
df1 = pd.DataFrame({'key':['a','b','c','d'],
                    'c1':[1,2,3,4]})
df2 = pd.DataFrame({'key':['a','b','c','e'],
                    'c2':[5,6,7,8,]})
display(df1)
display(df2)

In [ ]:
df1.merge(right=df2) # how='inner' : 양쪽 다 일치하는 데이터만 남김

In [ ]:
df1.merge(right=df2, how='left') # 왼쪽 데이터만 남기고 right에 매칭되는 것만 병합(left가 기준)


In [ ]:
df1.merge(df2, how='outer') # 양쪽 모두 다 남김

In [ ]:
df3 = pd.DataFrame({'key3':['a','b','c','d'],
                    'c1':[1,2,3,4]})
df4 = pd.DataFrame({'key4':['a','b','c','e'],
                    'c2':[5,6,7,8,]})
display(df3)
display(df4)

In [ ]:
df3.merge(right=df4, left_on='key3', right_on='key4', how='inner')

In [ ]:
df3.merge(right=df4, left_on='key3', right_on='key4', how='outer')

In [ ]:
df3.merge(df4, left_index=True, right_index=True) # pd.concat([df3,df4], 1) 와 유사

## 6.2 concat() 을 이용한 연결
- pd.concat( [df1,df2], axis)
    * axis=0(기본값) : 위아래로 연결
    * axis=1 : 좌우로 연결

In [ ]:
df1 = pd.DataFrame({'key':['a','b','c','d'],
                    'c1':[1,2,3,4]})
df2 = pd.DataFrame({'key':['a','b','c','e'],
                    'c1':[5,6,7,8,]})
pd.concat( [df1,df2], axis=0).reset_index(drop=True)

In [ ]:
# 좌우 연결
df3 = pd.DataFrame({'key3':['a','b','c','d'],
                    'c1':[1,2,3,4]})
df4 = pd.DataFrame({'key4':['a','b','c','e'],
                    'c2':[5,6,7,8,]})
pd.concat([df3,df4], axis=1)

# 7절 정렬(행이름, 열이름, 값에 의한 정렬)
- '100' < '9'
- df.sort_index(axis) : 행(axis=0) 또는 열(axis=1) 이름으로 정렬
- df.sort_values(by=정렬기준이 될 열이름, acending=T/F, inplace=T/F) : 값에 의한 정렬

In [ ]:
member = member.drop(['age'], axis=1)
member

In [ ]:
member = pd.read_csv('data/ch11_member.csv', comment='#')
member.index = ['동','서','남','북','중']
member

## 7.1 행이름 정렬

In [ ]:
member.sort_index(axis=0)
member.sort_index(axis='rows', inplace=True ) # ascending=True 기본값(오름차순)
member

## 7.2 열이름으로 정렬

In [ ]:
member.sort_index(axis=1)
member.sort_index(axis='columns', inplace=True) # ascending=True 기본값(오름차순)
member

## 7.3 값에 의한 정렬

In [ ]:
member.sort_values(by='Age') # 'Age' 열 데이터 기준으로 오름차순 정렬

In [ ]:
member.sort_values(by='Age', ascending=False, inplace=True) # 내림차순 정렬
member

In [ ]:
# 'Address' 기준으로 오름차순 정렬, 'Address' 가 같으면 'Age'
member.sort_values(by=['Address','Age'])

In [ ]:
# 'Address' 기준으로 오름차순 정렬, 'Address' 가 같으면 'Age' 내림차순
member.sort_values(by=['Address','Age'],
                   ascending=[True,False],
                   inplace=True)
member

In [ ]:
# iris 데이터 셋
# (1) sepal_length값 기준(sepal_length같으면 sepal_width기준) 내림차순 정렬 적용(iris에 적용)
# (2) 행이름(index) 기준 정렬 적용(iris에 적용)
# (3) 열이름(column) 기준 정렬한 내용을 출력(iris에 적용 X)
from seaborn import load_dataset
iris = load_dataset('iris')
iris.head(1)

In [ ]:
# (1) sepal_length값 기준(sepal_length같으면 sepal_width기준) 내림차순 정렬 적용(iris에 적용)
iris.sort_values(by=['sepal_length','sepal_width'],
                 ascending=False,
                 inplace=True)
iris

In [ ]:
# (2) 행이름(index) 기준 정렬 적용(iris에 적용)
iris.sort_index(axis=0)
iris.sort_index(axis='rows', inplace=True ) # ascending=True 기본값(오름차순)
iris

In [ ]:
# (3) 열이름(column) 기준 정렬한 내용을 출력(iris에 적용 X)
iris.sort_index(axis=1)
iris.sort_index(axis='columns', inplace=True) # ascending=True 기본값(오름차순)
iris

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

# 8절. 기초 통계 분석
    * 판다스 : 기초통계 / statsmodels : 난이도 있는 통계
- count : NaN을 제외한 갯수
- min
- max
- sum
- cumsum : 누적합
- cumprod : 누적곱
- mean : 평균
- rolling(n).mean() : 데이터 변동(노이즈)가 있을 때, 추세(패턴)을 부드럽게 보고 싶을 때
- median : 중앙값(50% 지점)
- var : 분산
- std : 표준편차
- qunantile : 분위수 - 0사분위수, 1사분위수(25%), 2사분위수(50%), 3사분위수(75%), 4사분위수(100%)
    IQR = Q3-Q1
    Q1 - 1.5*IQR ~ Q3+1.5*IQR
- describe : 요약통계량
- corr : 상관관계 (계수)

In [ ]:
from statsmodels.api import datasets
iris = datasets.get_rdataset('iris').data
iris.columns = [col.lower().replace('.','_') for col in iris.columns]
iris.columns

## 8.1 min, max, mean, std...

In [ ]:
iris.min(axis=0) # 열별 최소값(문자필드는 코드값이 작은 것)

In [ ]:
iris.median(axis=0, numeric_only=True) # 행들의 중수(열별 중위수)
# 평균, 중위수, 표준편차, 분산 ... : 숫자필드
# numeric_only=True : numeric 만 적용 

In [ ]:
X = iris.iloc[:,:-1]
X.sample()

In [ ]:
X.std(axis=0)

In [ ]:
X.mean(axis=1)

In [ ]:
# quantile : 데이터프레임이나 시리즈에서 사분위수
# interpolation='nearest' : 정확한 구간의 값이 없을 경우 가까운 데이터를 출력
df = pd.DataFrame(data=[1,3,4,7,10], columns=['value'])
df['value'].quantile(q=[0, 0.25, 0.5, 0.75, 1.], interpolation='nearest')


In [ ]:
# interpolation='midpoint' : 정확한 구간을 출력
df['value'].quantile(q=[0, 0.3, 0.55, 0.75, 1.], interpolation='midpoint')

In [ ]:
quant = X.quantile(q=[0, 0.25, 0.5, 0.75, 1])
quant

In [ ]:
min = quant.iloc[0, 0]
max = quant.iloc[4, 0]
q1 = quant.iloc[1, 0]
q3 = quant.iloc[3, 0]
min < q1-1.5*(q3-q1), max > q3 + 1.5 * (q3-q1)

In [ ]:
X.shape, X.count(axis=0) # 결측치를 제외한 데이터 갯수

In [ ]:
X.rolling(5).sum().iloc[4:]

## 8.2 요약 통계량


In [ ]:
# 1) 기본 요약 통계량
iris.describe() # 숫자열과 문자열이 같이 있을 경우 : 숫자열만

In [ ]:
# 문자열 요약통계량 : 데이터갯수, 데이터종류(unique), 최빈데이터(top), 최빈데이터갯수(freq)
iris['species'].describe()

In [ ]:
# 2) describe() 의 include와 exclude 매개변수
df = pd.DataFrame(data={'a':[1,2,3]*2,
                        'b':[2., 1]*3,
                        'c':['aaa']*5+['bbb'],
                        'd':[True,False]*3})
display(df)
df.info()

In [ ]:
df.describe() # 숫자열만 기본 요약 통계량

In [ ]:
df.describe(include=['float64','bool'])

In [ ]:
df.describe(include='all')

In [ ]:
df.describe(exclude='bool')

In [ ]:
df['c'].unique() # 특정 컬럼의 고유 값들의 종류

In [ ]:
df['c'].value_counts()

In [ ]:
df.select_dtypes(include=object) # object 형 컬럼만 추출

In [ ]:
df.select_dtypes(exclude=object)

## 8.3 공분산, 상관계수

In [ ]:
X

In [ ]:
X.cov()

In [ ]:
# -1 <= 상관계수 <= 1
X.corr()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(18,6))
sns.heatmap(X.corr(), vmin=-1, vmax=1, annot=True, fmt='.2f', cmap='Greens')
plt.show()